In [ ]:
import tensorflow as tf
import numpy as np
import os
import sys

project_root = os.path.abspath(os.path.join(".."))  # One level up from current script
if project_root not in sys.path:
    sys.path.append(project_root)

from keras.src.metrics.accuracy_metrics import accuracy
from src.model_loader import ModelLoader
from src.data_loader import DataLoader
from src.utils import get_class_weigths, get_confusion_matrix, get_classification_report

print(tf.__version__)

2.19.0


In [6]:
def is_valid_image(path):
    try:
        img_bytes = tf.io.read_file(path)
        decoded_img = tf.io.decode_image(img_bytes)
        return True
    except tf.errors.InvalidArgumentError as e:
        print(f"Found bad path {path}...{e}")
        return False

def clean_invalid_images(datasets_base_path):
    for root, dirs, files in os.walk(datasets_base_path):
        for file in files:
            if file.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
                image_path = os.path.join(root, file)
                if not is_valid_image(image_path):
                    print(f"Removing invalid image: {image_path}")
                    os.remove(image_path)

clean_invalid_images("../datasets")

2025-04-11 10:13:27.731241: W tensorflow/core/lib/png/png_io.cc:89] PNG warning: iCCP: known incorrect sRGB profile


Found bad path ../datasets/Painting/painting_02662.jpg...{{function_node __wrapped__DecodeImage_device_/job:localhost/replica:0/task:0/device:CPU:0}} Input size should match (header_size + row_size * abs_height) but they differ by 2 [Op:DecodeImage] name: 
Removing invalid image: ../datasets/Painting/painting_02662.jpg


2025-04-11 10:14:16.158706: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: INVALID_ARGUMENT: Input size should match (header_size + row_size * abs_height) but they differ by 2
Corrupt JPEG data: 419 extraneous bytes before marker 0xd9


In [7]:
# Préparation des datasets
data_loader = DataLoader(batch_size=64)

datasets = {
    "binary_nocw": data_loader.load_binary_dataset(
        positive_class="Photo", negative_classes=["Painting", "Text", "Schematics", "Sketch"]
    ),
    "binary_cw": data_loader.load_binary_dataset(
        positive_class="Photo", negative_classes=["Painting", "Text", "Schematics", "Sketch"], class_weights=True
    ),
    "multiclass_nocw": data_loader.load_multiclass_dataset(
        class_folders=["Photo", "Painting", "Text", "Schematics", "Sketch"]
    ),
    "multiclass_cw": data_loader.load_multiclass_dataset(
        class_folders=["Photo", "Painting", "Text", "Schematics", "Sketch"], class_weights=True
    ),
}

# CNN_HARD

In [ ]:
cnn_hard_loader = ModelLoader(model_name="CNN_HARD")

history_cnn_all_ds = {}
all_model_cnn = {}

with tf.device("/gpu:0"):
    for dataset_name, (train_data, val_data, test_data) in datasets.items():
        print(f"Model Type : CNN_HARD")
        print(f"Dataset name : {dataset_name}")


        if 'binary' in dataset_name:
            cnn_hard_model = cnn_hard_loader.create_model_CNN_hard(show_summary=False)
            all_model_cnn[f"{dataset_name}"] = cnn_hard_model
        elif 'multiclass' in dataset_name:
            cnn_hard_model = cnn_hard_loader.create_model_CNN_hard(show_summary=False, num_classes=5)
            all_model_cnn[f"{dataset_name}"] = cnn_hard_model

        if "nocw" in dataset_name:
            history_cnn = cnn_hard_model.fit(
                train_data,
                validation_data=val_data,
                epochs=10,
                verbose=2,
                callbacks=[cnn_hard_loader.get_tensorboard_callback(), cnn_hard_loader.get_early_stopping(), cnn_hard_loader.get_model_checkpoint()],
            )
            history_cnn_all_ds[f"{dataset_name}"] = history_cnn
        else:
            history_cnn = cnn_hard_model.fit(
                train_data,
                validation_data=val_data,
                epochs=10,
                verbose=2,
                class_weight=get_class_weigths(train_data, val_data),
                callbacks=[cnn_hard_loader.get_tensorboard_callback(), cnn_hard_loader.get_early_stopping(), cnn_hard_loader.get_model_checkpoint()],
            )
            history_cnn_all_ds[f"{dataset_name}"] = history_cnn

Epoch 1/10


/Users/tanguydumontier/PycharmProjects/CESI_DS/venv/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2025-04-11 11:34:41.306177: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.
2025-04-11 11:35:08.689911: W tensorflow/core/lib/png/png_io.cc:89] PNG warning: iCCP: known incorrect sRGB profile
Corrupt JPEG data: 419 extraneous bytes before marker 0xd9


400/400 - 71s - 179ms/step - accuracy: 0.6049 - loss: 18.4665 - val_accuracy: 0.7483 - val_loss: 21.3991
Epoch 2/10


2025-04-11 11:36:17.522269: W tensorflow/core/lib/png/png_io.cc:89] PNG warning: iCCP: known incorrect sRGB profile
Corrupt JPEG data: 419 extraneous bytes before marker 0xd9


400/400 - 70s - 175ms/step - accuracy: 0.6413 - loss: 20.0367 - val_accuracy: 0.7739 - val_loss: 3.0115
Epoch 3/10


...

828/828 - 362s - 438ms/step - accuracy: 0.2415 - loss: -2.0178e+18 - val_accuracy: 0.2415 - val_loss: -1.5104e+18
Epoch 9/10


2025-04-11 13:04:51.598949: W tensorflow/core/lib/png/png_io.cc:89] PNG warning: iCCP: known incorrect sRGB profile


828/828 - 368s - 444ms/step - accuracy: 0.2415 - loss: -3.2630e+18 - val_accuracy: 0.2415 - val_loss: -2.3660e+18
Epoch 10/10


2025-04-11 13:10:59.002459: W tensorflow/core/lib/png/png_io.cc:89] PNG warning: iCCP: known incorrect sRGB profile


828/828 - 334s - 404ms/step - accuracy: 0.2415 - loss: -4.9686e+18 - val_accuracy: 0.2415 - val_loss: -3.5323e+18


In [7]:
print(history_cnn_all_ds['multiclass_cw'].history)

{'accuracy': [0.24129991233348846, 0.24152638018131256, 0.24152638018131256, 0.24152638018131256, 0.24152638018131256, 0.24152638018131256, 0.24152638018131256, 0.24152638018131256, 0.24152638018131256, 0.24152638018131256], 'loss': [-53755022147584.0, -3137497768919040.0, -2.649408773344461e+16, -1.0405982485715354e+17, -2.829722023100416e+17, -6.187239864103076e+17, -1.165019018730406e+18, -2.0178337723381187e+18, -3.2630426479833907e+18, -4.968591341094175e+18], 'val_accuracy': [0.2415459007024765, 0.2415459007024765, 0.2415459007024765, 0.2415459007024765, 0.2415459007024765, 0.2415459007024765, 0.2415459007024765, 0.2415459007024765, 0.2415459007024765, 0.2415459007024765], 'val_loss': [-196905829138432.0, -5358189134479360.0, -3.1207335451951104e+16, -1.018514901125038e+17, -2.474345777510482e+17, -5.0241919261238886e+17, -9.06143610175488e+17, -1.5103723224799642e+18, -2.365976124770484e+18, -3.532324315619918e+18]}


# RES_NET

In [8]:
res_net_loader = ModelLoader(model_name="RES_NET")

history_resnet_all_ds = {}
all_model_resnet = {}

for dataset_name, (train_data, val_data, test_data) in datasets.items():
    print(f"Model Type : RES_NET")
    print(f"Dataset name : {dataset_name}")

    if 'binary' in dataset_name:
        res_net_model = res_net_loader.create_model_resnet50(show_summary=False)
        all_model_resnet[f"{dataset_name}"] = res_net_model
    elif 'multiclass' in dataset_name:
        res_net_model = res_net_loader.create_model_resnet50(show_summary=False, num_classes=5)
        all_model_resnet[f"{dataset_name}"] = res_net_model

    if "nocw" in dataset_name:
        history_resnet = res_net_model.fit(
            train_data,
            validation_data=val_data,
            epochs=10,
            verbose=1,
            callbacks=[res_net_loader.get_tensorboard_callback(), res_net_loader.get_early_stopping(), res_net_loader.get_model_checkpoint()],
        )
        history_resnet_all_ds[f"{dataset_name}"] = history_resnet
    else:
        history_resnet = res_net_model.fit(
            train_data,
            validation_data=val_data,
            epochs=10,
            verbose=1,
            class_weight=get_class_weigths(train_data, val_data),
            callbacks=[res_net_loader.get_tensorboard_callback(), res_net_loader.get_early_stopping(), res_net_loader.get_model_checkpoint()],
        )
        history_resnet_all_ds[f"{dataset_name}"] = history_resnet

Model Type : RES_NET
Dataset name : binary_nocw
Train data length : 200
Validation data length : 50
Dataset type : Binary
Epoch 1/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 186ms/step - accuracy: 0.6838 - loss: 0.5926

Corrupt JPEG data: 419 extraneous bytes before marker 0xd9
2025-04-11 14:03:27.404579: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 566231040 exceeds 10% of free system memory.
2025-04-11 14:03:27.706396: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 503316480 exceeds 10% of free system memory.
2025-04-11 14:03:28.693053: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 566231040 exceeds 10% of free system memory.
2025-04-11 14:03:29.426272: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 566231040 exceeds 10% of free system memory.


200/200 ━━━━━━━━━━━━━━━━━━━━ 64s 290ms/step - accuracy: 0.6840 - loss: 0.5923 - val_accuracy: 0.7736 - val_loss: 0.4639
Epoch 2/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 200ms/step - accuracy: 0.7653 - loss: 0.4782

Corrupt JPEG data: 419 extraneous bytes before marker 0xd9
2025-04-11 14:04:22.846303: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 566231040 exceeds 10% of free system memory.


200/200 ━━━━━━━━━━━━━━━━━━━━ 58s 288ms/step - accuracy: 0.7653 - loss: 0.4782 - val_accuracy: 0.7833 - val_loss: 0.4510
Epoch 3/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 179ms/step - accuracy: 0.7728 - loss: 0.4685

Corrupt JPEG data: 419 extraneous bytes before marker 0xd9


200/200 ━━━━━━━━━━━━━━━━━━━━ 65s 306ms/step - accuracy: 0.7728 - loss: 0.4685 - val_accuracy: 0.7852 - val_loss: 0.4398
Epoch 4/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 194ms/step - accuracy: 0.7747 - loss: 0.4602

Corrupt JPEG data: 419 extraneous bytes before marker 0xd9


200/200 ━━━━━━━━━━━━━━━━━━━━ 64s 271ms/step - accuracy: 0.7747 - loss: 0.4602 - val_accuracy: 0.7849 - val_loss: 0.4371
Epoch 5/10
122/200 ━━━━━━━━━━━━━━━━━━━━ 14s 180ms/step - accuracy: 0.7788 - loss: 0.4530

KeyboardInterrupt: 

# INCEPTION

In [ ]:

inception_loader = ModelLoader(model_name="INCEPTION")

history_inception_all_ds = {}
all_model_inception = {}

for dataset_name, (train_data, val_data, test_data) in datasets.items():
    print(f"Model Type : INCEPTION")
    print(f"Dataset name : {dataset_name}")

    if 'binary' in dataset_name:
        inception_model = inception_loader.create_model_with_inception(show_summary=False)
        all_model_inception[f"{dataset_name}"] = inception_model
    elif 'multiclass' in dataset_name:
        inception_model = inception_loader.create_model_with_inception(show_summary=False, num_classes=5)
        all_model_inception[f"{dataset_name}"] = inception_model

    if "nocw" in dataset_name:
        history_inception = inception_model.fit(
            train_data,
            validation_data=val_data,
            epochs=10,
            verbose=1,
            callbacks=[inception_loader.get_tensorboard_callback(), inception_loader.get_early_stopping(), inception_loader.get_model_checkpoint()],
        )
        history_inception_all_ds[f"{dataset_name}"] = history_inception
    else:
        history_inception = inception_model.fit(
            train_data,
            validation_data=val_data,
            epochs=10,
            verbose=1,
            class_weight=get_class_weigths(train_data, val_data),
            callbacks=[inception_loader.get_tensorboard_callback(), inception_loader.get_early_stopping(), inception_loader.get_model_checkpoint()],
        )
        history_inception_all_ds[f"{dataset_name}"] = history_inception

Model Type : INCEPTION
Dataset name : binary_nocw


AttributeError: 'Functional' object has no attribute 'get_tensorboard_callback'